# 面试问题：RAG 文档摄取链路怎样实现幂等更新、版本追踪和安全删除？

**一句话回答**：把 source event、规范化文档、chunk、embedding/index publication 都视为带版本的不可变制品；事件用 event ID 去重，文档用单调 source version 防旧覆盖，chunk ID 由 tenant/doc/version/offset/content 决定，删除写 tombstone 并传播到所有索引，最终通过 snapshot manifest 原子切换读流量。

下面用标准库实现文档合同、确定性 chunking、事件去重、outbox、增量索引、删除水位、引用 lineage 和可信 snapshot。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
from collections import Counter, defaultdict  # 导入本单元所需的依赖。
import hashlib, json, re, unicodedata  # 导入本单元所需的依赖。

SEED82=8201  # 计算并保存当前步骤的中间状态。
def sha82(text): return hashlib.sha256(text.encode()).hexdigest()  # 定义本节可复用的核心函数。
assert len(sha82("x"))==64  # 用受控断言验证关键不变量。
assert sha82("x")!=sha82("X")  # 用受控断言验证关键不变量。
assert unicodedata.normalize("NFKC","Ａ")=="A"  # 用受控断言验证关键不变量。

## 1. source event 与文档合同

upsert/delete 事件包含全局 event ID、tenant、doc ID、source version、event time 和 ACL。`source_version` 必须在同一文档内单调；event ID 解决消息重复，source version 解决乱序。两者缺一不可。

文档正文与 ACL 同样属于版本内容。ACL 改变即使正文未变也必须发布新版本，否则旧 chunk 可能继续对无权限用户可见。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Event82:  # 定义承载本节状态与行为的数据结构。
    event_id:str; op:str; tenant:str; doc_id:str; source_version:int; event_time:int; acl:tuple=(); content:str|None=None  # 计算并保存当前步骤的中间状态。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        if not self.event_id or self.op not in {"upsert","delete"} or not self.tenant or not self.doc_id or self.source_version<1: raise ValueError("event_contract")  # 按当前条件选择后续控制路径。
        if self.op=="upsert" and (not isinstance(self.content,str) or not self.content.strip() or not self.acl): raise ValueError("event_contract")  # 按当前条件选择后续控制路径。
        if self.op=="delete" and self.content is not None: raise ValueError("event_contract")  # 按当前条件选择后续控制路径。
e82=Event82("e1","upsert","t1","d1",1,10,("reader",),"标题。第一段内容。第二段内容。")  # 计算并保存当前步骤的中间状态。
assert e82.source_version==1 and e82.op=="upsert"  # 用受控断言验证关键不变量。
try: Event82("x","delete","t","d",1,0,(),"should-be-none"); raise AssertionError("invalid delete accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="event_contract"  # 捕获预期异常并验证失败分支。
assert Event82("e2","delete","t1","d1",2,20).content is None  # 用受控断言验证关键不变量。

## 2. 确定性规范化与 chunking

chunker 必须返回规范化文本中的 `[start,end)` offset，不能只返回字符串；引用、高亮和增量 diff 都依赖 offset。这里优先在中文句号/换行后切分，超过上限才硬切，并保留 overlap。相同输入和 chunker version 必须生成相同 ID。

normalization 会改变 offset 语义，因此原文、规范化文本和映射规则都应有版本。教学版只保存规范化 offset。

In [ ]:
def normalize82(text): return re.sub(r"[ \t]+"," ",unicodedata.normalize("NFKC",text).replace("\r\n","\n")).strip()  # 定义本节可复用的核心函数。
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Chunk82:  # 定义承载本节状态与行为的数据结构。
    chunk_id:str; tenant:str; doc_id:str; source_version:int; start:int; end:int; text:str; acl:tuple; content_sha:str  # 执行当前语句以推进本节示例。
def chunk82(event,max_chars=18,overlap=4,chunker_version="sentence-v1"):  # 定义本节可复用的核心函数。
    if event.op!="upsert" or not 0<=overlap<max_chars: raise ValueError("chunk_contract")  # 按当前条件选择后续控制路径。
    text=normalize82(event.content); chunks=[]; start=0  # 计算并保存当前步骤的中间状态。
    while start<len(text):  # 在终止条件满足前持续推进状态。
        cap=min(len(text),start+max_chars); boundary=cap  # 计算并保存当前步骤的中间状态。
        if cap<len(text):  # 按当前条件选择后续控制路径。
            options=[i+1 for i in range(start,cap) if text[i] in "。！？\n"]  # 计算并保存当前步骤的中间状态。
            if options and options[-1]>start+max_chars//2: boundary=options[-1]  # 按当前条件选择后续控制路径。
        piece=text[start:boundary]  # 计算并保存当前步骤的中间状态。
        raw="|".join([event.tenant,event.doc_id,str(event.source_version),str(start),str(boundary),chunker_version,sha82(piece)])  # 计算并保存当前步骤的中间状态。
        chunks.append(Chunk82(sha82(raw)[:24],event.tenant,event.doc_id,event.source_version,start,boundary,piece,event.acl,sha82(piece)))  # 执行当前语句以推进本节示例。
        if boundary==len(text): break  # 按当前条件选择后续控制路径。
        start=max(start+1,boundary-overlap)  # 计算并保存当前步骤的中间状态。
    return chunks  # 返回当前分支计算出的结果。
chunks82=chunk82(e82,10,2)  # 计算并保存当前步骤的中间状态。
assert chunks82 and chunks82[0].start==0 and chunks82[-1].end==len(normalize82(e82.content))  # 用受控断言验证关键不变量。
assert len({c.chunk_id for c in chunks82})==len(chunks82) and chunks82==chunk82(e82,10,2)  # 用受控断言验证关键不变量。
assert all(c.text==normalize82(e82.content)[c.start:c.end] for c in chunks82)  # 用受控断言验证关键不变量。

## 3. 幂等状态机与 transactional outbox

摄取事务原子更新 `latest_version`、不可变 chunk 表、tombstone 和 outbox。消息系统即使 at-least-once，重复 event ID 也不会重复建索引；旧 source version 返回 stale。相同版本不同 event/content 是冲突，需要人工调查，不能 last-write-wins。

真实数据库用唯一键和事务；这里用内存结构表达状态转换。

In [ ]:
class IngestionStore82:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.seen_events=set(); self.latest={}; self.chunks={}; self.active_by_doc=defaultdict(set); self.tombstones={}; self.outbox=[]  # 定义本节可复用的核心函数。
    def apply(self,event):  # 定义本节可复用的核心函数。
        key=(event.tenant,event.doc_id); old=self.latest.get(key,0)  # 计算并保存当前步骤的中间状态。
        if event.event_id in self.seen_events: return "duplicate"  # 按当前条件选择后续控制路径。
        if event.source_version<old: self.seen_events.add(event.event_id); return "stale"  # 按当前条件选择后续控制路径。
        if event.source_version==old: raise RuntimeError("same_version_conflict")  # 按当前条件选择后续控制路径。
        self.seen_events.add(event.event_id); self.latest[key]=event.source_version  # 计算并保存当前步骤的中间状态。
        previous=list(self.active_by_doc[key]); self.active_by_doc[key].clear()  # 计算并保存当前步骤的中间状态。
        for cid in previous: self.outbox.append(("delete_chunk",cid,event.source_version))  # 遍历输入元素以累积或检查结果。
        if event.op=="delete": self.tombstones[key]=event.source_version; self.outbox.append(("delete_doc",key,event.source_version)); return "deleted"  # 按当前条件选择后续控制路径。
        self.tombstones.pop(key,None)  # 执行当前语句以推进本节示例。
        for c in chunk82(event): self.chunks[c.chunk_id]=c; self.active_by_doc[key].add(c.chunk_id); self.outbox.append(("upsert_chunk",c.chunk_id,event.source_version))  # 遍历输入元素以累积或检查结果。
        return "upserted"  # 返回当前分支计算出的结果。
store82=IngestionStore82()  # 计算并保存当前步骤的中间状态。
assert store82.apply(e82)=="upserted" and store82.apply(e82)=="duplicate"  # 用受控断言验证关键不变量。
assert store82.latest[("t1","d1")]==1 and store82.active_by_doc[("t1","d1")]  # 用受控断言验证关键不变量。
stale_probe82=IngestionStore82()  # 计算并保存当前步骤的中间状态。
assert stale_probe82.apply(Event82("newer","upsert","t1","probe",2,10,("reader",),"新版"))=="upserted"  # 用受控断言验证关键不变量。
assert stale_probe82.apply(Event82("older","upsert","t1","probe",1,11,("reader",),"旧版"))=="stale"  # 用受控断言验证关键不变量。
try: store82.apply(Event82("conflict","upsert","t1","d1",1,12,("reader",),"不同内容")); raise AssertionError("same version accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="same_version_conflict"  # 捕获预期异常并验证失败分支。

## 4. outbox 驱动增量索引

index consumer 也必须幂等：每条 outbox record 有稳定 offset，consumer 保存 checkpoint。先写索引再推进 checkpoint，崩溃会重放但不会丢；反过来可能永久漏索引。索引记录保存 chunk version 和 ACL，查询只读已发布 snapshot。

下例用 token 倒排表表示目标索引，并处理 chunk tombstone。

In [ ]:
class TinyIndex82:  # 定义承载本节状态与行为的数据结构。
    def __init__(self): self.docs={}; self.postings=defaultdict(set); self.checkpoint=0  # 定义本节可复用的核心函数。
    def _remove(self,cid):  # 定义本节可复用的核心函数。
        old=self.docs.pop(cid,None)  # 计算并保存当前步骤的中间状态。
        if old:  # 按当前条件选择后续控制路径。
            for tok in set(re.findall(r"[\u4e00-\u9fff]|[a-z0-9]+",old.text.casefold())): self.postings[tok].discard(cid)  # 遍历输入元素以累积或检查结果。
    def consume(self,store,limit=None):  # 定义本节可复用的核心函数。
        end=len(store.outbox) if limit is None else min(len(store.outbox),self.checkpoint+limit)  # 计算并保存当前步骤的中间状态。
        while self.checkpoint<end:  # 在终止条件满足前持续推进状态。
            op,target,version=store.outbox[self.checkpoint]  # 计算并保存当前步骤的中间状态。
            if op=="upsert_chunk":  # 按当前条件选择后续控制路径。
                c=store.chunks[target]; self._remove(target); self.docs[target]=c  # 计算并保存当前步骤的中间状态。
                for tok in set(re.findall(r"[\u4e00-\u9fff]|[a-z0-9]+",c.text.casefold())): self.postings[tok].add(target)  # 遍历输入元素以累积或检查结果。
            elif op=="delete_chunk": self._remove(target)  # 按当前条件选择后续控制路径。
            elif op=="delete_doc": pass  # 按当前条件选择后续控制路径。
            self.checkpoint+=1  # 计算并保存当前步骤的中间状态。
    def search(self,token,tenant,principal):  # 定义本节可复用的核心函数。
        return sorted(cid for cid in self.postings[token] if self.docs[cid].tenant==tenant and principal in self.docs[cid].acl)  # 返回当前分支计算出的结果。
index82=TinyIndex82(); index82.consume(store82,1); partial82=index82.checkpoint; index82.consume(store82)  # 计算并保存当前步骤的中间状态。
assert partial82==1 and index82.checkpoint==len(store82.outbox)  # 用受控断言验证关键不变量。
assert set(index82.docs)==store82.active_by_doc[("t1","d1")]  # 用受控断言验证关键不变量。
assert index82.search("标","t1","reader") and index82.search("标","t1","outsider")==[]  # 用受控断言验证关键不变量。

## 5. 主存储与派生索引必须持续对账

“consumer 没报错”不代表索引完整。根据主存储 active chunk 集与索引 doc 集计算 missing/extra，并按 tenant/version 切片；missing 触发补写，extra 触发 tombstone。对账只比较 ID/摘要，避免扫描时泄露正文。

这里注入一个幽灵 chunk，证明对账能发现并在修复后恢复为空差异。

In [ ]:
def reconcile82(store,index):  # 定义本节可复用的核心函数。
    expected=set().union(*store.active_by_doc.values()) if store.active_by_doc else set()  # 计算并保存当前步骤的中间状态。
    actual=set(index.docs)  # 计算并保存当前步骤的中间状态。
    return {"missing":sorted(expected-actual),"extra":sorted(actual-expected)}  # 返回当前分支计算出的结果。
assert reconcile82(store82,index82)=={"missing":[],"extra":[]}  # 用受控断言验证关键不变量。
ghost82=next(iter(index82.docs.values())); index82.docs["ghost-id"]=ghost82  # 计算并保存当前步骤的中间状态。
assert reconcile82(store82,index82)["extra"]==["ghost-id"]  # 用受控断言验证关键不变量。
del index82.docs["ghost-id"]  # 执行当前语句以推进本节示例。
assert reconcile82(store82,index82)=={"missing":[],"extra":[]}  # 用受控断言验证关键不变量。

## 6. 更新、删除与旧引用

新版本先为旧 active chunk 产生 delete，再产生新 upsert。查询 snapshot 切换后旧 chunk 不可召回；但审计表可保留不可变 lineage。删除必须覆盖 sparse/dense/cache/reranker feature 等所有派生物，并用 deletion watermark 证明传播完成。

旧答案中的 citation 指向 `(doc_id,source_version,offset,content_sha)`；若当前版本不同，UI 应标记引用已过期，而不是静默指向新文本。

In [ ]:
update82=Event82("e2","upsert","t1","d1",2,20,("reader","admin"),"新标题。退款流程已经更新。")  # 计算并保存当前步骤的中间状态。
old_ids82=set(store82.active_by_doc[("t1","d1")]); assert store82.apply(update82)=="upserted"; new_ids82=set(store82.active_by_doc[("t1","d1")])  # 计算并保存当前步骤的中间状态。
assert old_ids82.isdisjoint(new_ids82) and store82.latest[("t1","d1")]==2  # 用受控断言验证关键不变量。
index82.consume(store82)  # 执行当前语句以推进本节示例。
assert not (old_ids82 & set(index82.docs)) and set(index82.docs)==new_ids82  # 用受控断言验证关键不变量。
deletion82=Event82("e3","delete","t1","d1",3,30)  # 计算并保存当前步骤的中间状态。
assert store82.apply(deletion82)=="deleted"; index82.consume(store82)  # 用受控断言验证关键不变量。
assert not index82.docs and store82.tombstones[("t1","d1")]==3  # 用受控断言验证关键不变量。

## 7. Snapshot publication 与回滚

读请求不能看到“半数 chunk 新、半数 chunk 旧”。构建者在隔离 namespace 消费到目标 outbox checkpoint，核对 doc/chunk/tombstone 数量与权限，再写 manifest；路由层原子切换 snapshot ID。回滚只切指针，不原地重写索引。

manifest 摘要覆盖 chunk IDs、版本水位和 chunker config，防止同名 snapshot 内容漂移。

In [ ]:
def snapshot82(store,index,snapshot_id):  # 定义本节可复用的核心函数。
    if index.checkpoint!=len(store.outbox): raise RuntimeError("index_not_caught_up")  # 按当前条件选择后续控制路径。
    body={"snapshot_id":snapshot_id,"checkpoint":index.checkpoint,"latest":sorted((list(k),v) for k,v in store.latest.items()),"active_chunks":sorted(index.docs),"tombstones":sorted((list(k),v) for k,v in store.tombstones.items()),"chunker":"sentence-v1"}  # 计算并保存当前步骤的中间状态。
    raw=json.dumps(body,ensure_ascii=False,sort_keys=True,separators=(",",":")); return {"body":body,"sha256":sha82(raw)}  # 计算并保存当前步骤的中间状态。
snap82=snapshot82(store82,index82,"snap-3")  # 计算并保存当前步骤的中间状态。
assert snap82["body"]["checkpoint"]==len(store82.outbox) and snap82["body"]["active_chunks"]==[]  # 用受控断言验证关键不变量。
assert len(snap82["sha256"])==64 and snap82["body"]["tombstones"][0][1]==3  # 用受控断言验证关键不变量。
forged82=json.loads(json.dumps(snap82)); forged82["body"]["checkpoint"]-=1  # 计算并保存当前步骤的中间状态。
assert sha82(json.dumps(forged82["body"],ensure_ascii=False,sort_keys=True,separators=(",",":")))!=forged82["sha256"]  # 用受控断言验证关键不变量。

## 8. Replay、对账与故障注入

最重要的恢复测试是从空状态重放全部 source events，结果必须与在线状态等价。还要注入：消息重复、乱序、consumer 崩溃、同版本冲突、ACL 变更、删除后迟到 upsert。对账指标包括 source 最新文档数、active chunk 数、索引文档数和 tombstone 水位。

replay 日志本例手工给出；生产中来自不可变 CDC/WAL，不能只依赖当前数据库行。

In [ ]:
replay82=IngestionStore82()  # 计算并保存当前步骤的中间状态。
for event in [e82,e82,update82,deletion82]: replay82.apply(event)  # 遍历输入元素以累积或检查结果。
replay_index82=TinyIndex82(); replay_index82.consume(replay82)  # 计算并保存当前步骤的中间状态。
assert replay82.latest==store82.latest and replay82.tombstones==store82.tombstones  # 用受控断言验证关键不变量。
assert set(replay_index82.docs)==set(index82.docs)==set()  # 用受控断言验证关键不变量。
assert replay_index82.checkpoint==len(replay82.outbox) and len(replay82.seen_events)==3  # 用受控断言验证关键不变量。
print({"events":len(store82.seen_events),"outbox":len(store82.outbox),"checkpoint":index82.checkpoint,"tombstones":len(store82.tombstones)})  # 执行当前语句以推进本节示例。

## 9. 面试收束、参考与练习

完整回答：source/event 合同 → 确定性 chunk+lineage → event 去重与版本防乱序 → transactional outbox → 幂等多索引 consumer → tombstone 水位 → snapshot 原子发布/回滚 → replay 对账。只讲“文档切块后 embedding”远远不够。

练习：添加 PDF page/box provenance；实现只重算变化 chunk 的 diff；模拟 dense index 比 sparse index 落后一版时的路由门禁；设计 GDPR 删除证明。

参考：[Transactional Outbox pattern](https://microservices.io/patterns/data/transactional-outbox.html)、[RFC 9110 条件请求与版本语义](https://www.rfc-editor.org/rfc/rfc9110)、[RAG 原论文](https://arxiv.org/abs/2005.11401)。